# Clean VIXY Options Data

This notebook cleans the raw `VIXY` options dataset according to the project plan:

- load the raw options chain
- clean obvious bad rows
- compute `DTE` and option mid-prices
- keep the `25 <= DTE <= 35` contracts
- choose the contract closest to `30 DTE`
- choose the strike with delta closest to `0.5` for calls and puts
- export cleaned CSV files for modeling and backtesting


In [6]:
from pathlib import Path

import pandas as pd

In [7]:
project_dir = Path.cwd()
if not (project_dir / "i7k7ktpw91k9qvhq.csv").exists():
    project_dir = project_dir / "Project"

raw_file = project_dir / "i7k7ktpw91k9qvhq.csv"
clean_file = project_dir / "vixy_options_clean.csv"
candidate_file = project_dir / "vixy_30d_candidates.csv"
panel_file = project_dir / "vixy_30d_atm_panel.csv"

df_raw = pd.read_csv(raw_file, low_memory=False)

for date_col in ["date", "exdate", "last_date"]:
    df_raw[date_col] = pd.to_datetime(df_raw[date_col], errors="coerce")

numeric_cols = [
    "strike_price",
    "best_bid",
    "best_offer",
    "volume",
    "open_interest",
    "impl_volatility",
    "delta",
    "gamma",
    "vega",
    "theta",
    "contract_size",
    "forward_price",
]

for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

print(df_raw.shape)
df_raw.head()

(222622, 38)


,secid,date,symbol,symbol_flag,exdate,last_date,cp_flag,strike_price,best_bid,best_offer,...,sic,index_flag,exchange_d,class,issue_type,industry_group,issuer,div_convention,exercise_style,am_set_flag
0,145765,2024-08-29,VIXY 240830C10000,1,2024-08-30,2024-08-29,C,10000,0.00,3.10,...,NaN,0,32,NaN,%,NaN,PROSHARES TRUST,NaN,A,NaN
1,145765,2024-08-29,VIXY 240830C10500,1,2024-08-30,2024-08-28,C,10500,0.00,1.15,...,NaN,0,32,NaN,%,NaN,PROSHARES TRUST,NaN,A,NaN
2,145765,2024-08-29,VIXY 240830C11000,1,2024-08-30,2024-08-29,C,11000,0.30,0.45,...,NaN,0,32,NaN,%,NaN,PROSHARES TRUST,NaN,A,NaN
3,145765,2024-08-29,VIXY 240830C11500,1,2024-08-30,2024-08-29,C,11500,0.10,0.15,...,NaN,0,32,NaN,%,NaN,PROSHARES TRUST,NaN,A,NaN
4,145765,2024-08-29,VIXY 240830C12000,1,2024-08-30,2024-08-29,C,12000,0.05,0.10,...,NaN,0,32,NaN,%,NaN,PROSHARES TRUST,NaN,A,NaN


In [3]:
df_clean = df_raw.copy()

df_clean["strike"] = df_clean["strike_price"] / 1000
df_clean["dte"] = (df_clean["exdate"] - df_clean["date"]).dt.days
df_clean["mid_price"] = (df_clean["best_bid"] + df_clean["best_offer"]) / 2
df_clean["bid_ask_spread"] = df_clean["best_offer"] - df_clean["best_bid"]
df_clean["abs_delta"] = df_clean["delta"].abs()
df_clean["delta_distance_50"] = (df_clean["abs_delta"] - 0.5).abs()

df_clean = df_clean[
    df_clean["date"].notna()
    & df_clean["exdate"].notna()
    & df_clean["cp_flag"].isin(["C", "P"])
    & df_clean["best_bid"].notna()
    & df_clean["best_offer"].notna()
    & (df_clean["best_bid"] >= 0)
    & (df_clean["best_offer"] >= df_clean["best_bid"])
    & df_clean["strike"].notna()
    & (df_clean["strike"] > 0)
    & df_clean["contract_size"].fillna(0).gt(0)
    & df_clean["dte"].notna()
    & (df_clean["dte"] >= 0)
].copy()

df_clean = df_clean.sort_values(["date", "exdate", "cp_flag", "strike"]).reset_index(drop=True)

print("Raw rows:", len(df_raw))
print("Clean rows:", len(df_clean))
df_clean[["date", "exdate", "cp_flag", "strike", "best_bid", "best_offer", "delta", "dte"]].head()

NameError: name 'df_raw' is not defined

In [2]:
df_candidates = df_clean[df_clean["dte"].between(25, 35)].copy()
df_candidates = df_candidates[df_candidates["delta"].notna()].copy()
df_candidates["dte_distance_30"] = (df_candidates["dte"] - 30).abs()

df_candidates = df_candidates.sort_values(
    ["date", "dte_distance_30", "exdate", "cp_flag", "delta_distance_50", "strike"]
).reset_index(drop=True)

print("30-day candidate rows:", len(df_candidates))
print("Unique trade dates in candidates:", df_candidates["date"].nunique())
df_candidates[["date", "exdate", "cp_flag", "strike", "delta", "dte", "dte_distance_30"]].head(10)

NameError: name 'df_clean' is not defined

In [ ]:
chosen_expiry = (
    df_candidates.sort_values(["date", "dte_distance_30", "exdate"])
    .groupby("date", as_index=False)
    .first()[["date", "exdate"]]
)

df_panel = df_candidates.merge(chosen_expiry, on=["date", "exdate"], how="inner")
df_panel = df_panel.sort_values(["date", "cp_flag", "delta_distance_50", "strike"])
df_panel = df_panel.groupby(["date", "cp_flag"], as_index=False).first()
df_panel = df_panel.sort_values(["date", "cp_flag"]).reset_index(drop=True)

df_panel["roll_id"] = df_panel.groupby("cp_flag")["exdate"].transform(lambda s: s.ne(s.shift()).cumsum())
df_panel["is_new_roll"] = df_panel.groupby("cp_flag")["exdate"].transform(lambda s: s.ne(s.shift()))

panel_cols = [
    "date",
    "exdate",
    "cp_flag",
    "symbol",
    "ticker",
    "optionid",
    "strike",
    "best_bid",
    "best_offer",
    "mid_price",
    "impl_volatility",
    "delta",
    "gamma",
    "vega",
    "theta",
    "volume",
    "open_interest",
    "dte",
    "dte_distance_30",
    "delta_distance_50",
    "roll_id",
    "is_new_roll",
]

df_panel = df_panel[panel_cols]

print("Target panel rows:", len(df_panel))
print("Target panel dates:", df_panel["date"].nunique())
print(df_panel["cp_flag"].value_counts())
df_panel.head(10)

Target panel rows: 158
Target panel dates: 79
cp_flag
C    79
P    79
Name: count, dtype: int64


,date,exdate,cp_flag,symbol,ticker,optionid,strike,best_bid,best_offer,mid_price,...,gamma,vega,theta,volume,open_interest,dte,dte_distance_30,delta_distance_50,roll_id,is_new_roll
0,2024-09-13,2024-10-18,C,VIXY 241018C13000,VIXY,163777973,13.0,0.85,1.00,0.925,...,0.119373,1.464171,-6.558119,19,289,35,5,0.061025,1,True
1,2024-09-13,2024-10-18,P,VIXY 241018P13000,VIXY,163777994,13.0,1.60,2.20,1.900,...,0.118187,1.463758,-6.872662,1,126,35,5,0.053636,1,True
2,2024-09-16,2024-10-18,C,VIXY 241018C13000,VIXY,163777973,13.0,0.85,1.05,0.950,...,0.118833,1.410278,-7.094286,92,298,32,2,0.055138,1,False
3,2024-09-16,2024-10-18,P,VIXY 241018P13000,VIXY,163777994,13.0,1.75,2.00,1.875,...,0.117218,1.407701,-7.772624,0,126,32,2,0.046608,1,False
4,2024-09-17,2024-10-18,C,VIXY 241018C13000,VIXY,163777973,13.0,1.00,1.25,1.125,...,0.108836,1.409439,-7.144349,59,377,31,1,0.031218,1,False
5,2024-09-17,2024-10-18,P,VIXY 241018P13000,VIXY,163777994,13.0,1.55,2.00,1.775,...,0.117374,1.395711,-9.216777,0,126,31,1,0.034009,1,False
6,2024-09-18,2024-10-18,C,VIXY 241018C13000,VIXY,163777973,13.0,0.95,1.10,1.025,...,0.117159,1.388387,-7.653844,1102,408,30,0,0.037401,1,False
7,2024-09-18,2024-10-18,P,VIXY 241018P13000,VIXY,163777994,13.0,1.55,2.15,1.850,...,0.111997,1.386816,-8.856990,0,126,30,0,0.024309,1,False
8,2024-09-19,2024-10-18,C,VIXY 241018C12000,VIXY,163777972,12.0,0.85,1.00,0.925,...,0.154935,1.320157,-6.043129,377,1254,29,1,0.007851,1,False
9,2024-09-19,2024-10-18,P,VIXY 241018P12000,VIXY,163777993,12.0,1.05,1.25,1.150,...,0.151835,1.318656,-6.785963,44,174,29,1,0.014469,1,False


In [ ]:
df_clean.to_csv(clean_file, index=False)
df_candidates.to_csv(candidate_file, index=False)
df_panel.to_csv(panel_file, index=False)

print("Wrote:")
print(clean_file)
print(candidate_file)
print(panel_file)

Wrote:
c:\School\Junior SP\Equity\Project\vixy_options_clean.csv
c:\School\Junior SP\Equity\Project\vixy_30d_candidates.csv
c:\School\Junior SP\Equity\Project\vixy_30d_atm_panel.csv
